# # LSTM Model: Walk-Forward Validation for Bitcoin
# # Python version 3.11+

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import time  

# Data and Preprocessing
import yfinance as yf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split  

# LSTM / ANN
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout  
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import callbacks  
import keras_tuner as kt  

# Visualization
import plotly.graph_objects as go
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

## 2. Configuration

In [ ]:
# --- Defined Parameters ---
ticker = "BTC-USD" 
start_date = "2017-11-09"
end_date = "2025-01-01"

# Define Train/Test Split Ratio for the *initial* training phase
train_split_ratio = 0.80
# Define Validation Split Ratio *within* the initial training data for tuning
validation_split_ratio_for_tuning = 0.20 # 20% of initial train data for validation

# LSTM Network Parameters
look_back = 60 

# Keras Tuner Configuration
MAX_TRIALS = 10 
EXECUTIONS_PER_TRIAL = 2 
TUNER_EPOCHS = 50 
TUNER_PATIENCE = 5 

# Final Training Configuration
FINAL_TRAINING_EPOCHS = 100
FINAL_TRAINING_PATIENCE = 10 

RETRAIN_FREQUENCY = 0
RETRAIN_EPOCHS = 5
FINAL_TRAINING_BATCH_SIZE = 32 # Default batch size if not tuned

## 3. Data Loading and Preparation

In [3]:
print(f"--- Loading Data for {ticker} ---")
try:
    df_full = yf.download(tickers=[ticker], start=start_date, end=end_date, progress=False)
    if df_full.empty: raise ValueError(f"No data downloaded for {ticker}.")
    if 'Close' not in df_full.columns: raise ValueError(f"'Close' column not found.")
    df_full = df_full[['Close']].copy()
    df_full = df_full.asfreq('D')
    df_full.ffill(inplace=True)
    df_full.dropna(inplace=True)
    if df_full.empty: raise ValueError(f"Data became empty after processing.")
    print(f"Loaded {len(df_full)} data points for {ticker} from {df_full.index.min()} to {df_full.index.max()}.")
except Exception as e:
    raise ValueError(f"Failed to load data for {ticker}: {e}")

# Split Data into initial train+val and test sets
n_total = len(df_full)
n_train_val = int(train_split_ratio * n_total) 
n_test = n_total - n_train_val
# Data for initial training and tuning
train_val_data_df = df_full[:n_train_val] 
# Data for walk-forward evaluation
test_data_df = df_full[n_train_val:] 

print(f"\nInitial Train+Validation Data: {n_train_val} points ({train_val_data_df.index.min().strftime('%Y-%m-%d')} to {train_val_data_df.index.max().strftime('%Y-%m-%d')})")
print(f"Test Data (for walk-forward): {n_test} points ({test_data_df.index.min().strftime('%Y-%m-%d')} to {test_data_df.index.max().strftime('%Y-%m-%d')})")

--- Loading Data for BTC-USD ---
YF.download() has changed argument auto_adjust default to True
Loaded 1827 data points for BTC-USD from 2020-01-01 00:00:00 to 2024-12-31 00:00:00.

Initial Train+Validation Data: 1461 points (2020-01-01 to 2023-12-31)
Test Data (for walk-forward): 366 points (2024-01-01 to 2024-12-31)


## 4. Scaling (Fit on Initial Train Portion Only)

In [4]:
print("\n--- Scaling Data ---")
# Further split train_val_data into train and validation for tuning scaler fit
n_val_tune = int(validation_split_ratio_for_tuning * n_train_val)
n_train_tune = n_train_val - n_val_tune

train_tune_values_for_scaler = train_val_data_df['Close'].values[:n_train_tune].reshape(-1, 1)

scaler = MinMaxScaler(feature_range=(0, 1))
# Fit scaler ONLY on the strict initial training portion (excluding tuning validation)
scaler.fit(train_tune_values_for_scaler)
print("Scaler fitted on initial training portion (excluding validation).")

# Transform the entire dataset using the fitted scaler
scaled_data = scaler.transform(df_full['Close'].values.reshape(-1, 1))

# Separate scaled train/validation portions for tuning
scaled_train_tune_data = scaled_data[:n_train_tune]
scaled_val_tune_data = scaled_data[n_train_tune:n_train_val]


--- Scaling Data ---
Scaler fitted on initial training portion (excluding validation).


## 5. Sequence Generation Function

In [5]:
def create_sequences(data, look_back):
    X, Y = [], []
    if data.ndim == 1: data = data.reshape(-1, 1)
    for i in range(look_back, len(data)):
        X.append(data[i - look_back:i, 0])
        Y.append(data[i, 0])
    return np.array(X), np.array(Y)

## 6. Prepare Data for Tuning

In [6]:
print("\n--- Preparing Sequences for Tuning ---")
# Create sequences for the tuning training set
x_train_tune, y_train_tune = create_sequences(scaled_train_tune_data, look_back)
x_train_tune = np.reshape(x_train_tune, (x_train_tune.shape[0], x_train_tune.shape[1], 1))

# Create sequences for the tuning validation set
# Need data overlap for lookback
val_tune_data_for_seq = scaled_data[n_train_tune - look_back : n_train_val, :]
x_val_tune, y_val_tune = create_sequences(val_tune_data_for_seq, look_back)
x_val_tune = np.reshape(x_val_tune, (x_val_tune.shape[0], x_val_tune.shape[1], 1))

print('x_train_tune shape:', x_train_tune.shape)
print('y_train_tune shape:', y_train_tune.shape)
print('x_val_tune shape:', x_val_tune.shape)
print('y_val_tune shape:', y_val_tune.shape)


--- Preparing Sequences for Tuning ---
x_train_tune shape: (1109, 60, 1)
y_train_tune shape: (1109,)
x_val_tune shape: (292, 60, 1)
y_val_tune shape: (292,)


## 7. LSTM Model Building Function for Keras Tuner

In [7]:
def build_model(hp):
    """Builds LSTM model with tunable hyperparameters."""
    model = Sequential()
    model.add(LSTM(units=hp.Int('units_1', min_value=32, max_value=128, step=32),
                   return_sequences=True,
                   input_shape=(look_back, 1)))
    model.add(Dropout(rate=hp.Float('dropout_1', min_value=0.0, max_value=0.3, step=0.1)))
    model.add(LSTM(units=hp.Int('units_2', min_value=32, max_value=128, step=32),
                   return_sequences=False))
    model.add(Dropout(rate=hp.Float('dropout_2', min_value=0.0, max_value=0.3, step=0.1)))
    model.add(Dense(units=hp.Int('dense_units', min_value=16, max_value=64, step=16), activation='relu')) # Added activation
    model.add(Dense(1)) # Output layer

    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])
    
    hp_batch_size = hp.Choice('batch_size', values=[16, 32, 64])

    model.compile(optimizer=Adam(learning_rate=hp_learning_rate),
                  loss='mean_squared_error')
    return model

## 8. Hyperparameter Tuning

In [8]:
print("\n--- Starting Hyperparameter Search with Keras Tuner ---")
tuner = kt.RandomSearch(
    build_model,
    objective='val_loss',
    max_trials=MAX_TRIALS,
    executions_per_trial=EXECUTIONS_PER_TRIAL,
    directory='keras_tuner_lstm_wf',  
    project_name=f'{ticker}_lstm_wf_tuning'
)

tuner_early_stopping = callbacks.EarlyStopping(monitor='val_loss', patience=TUNER_PATIENCE)

tuner.search(x_train_tune, y_train_tune,
             epochs=TUNER_EPOCHS,
             validation_data=(x_val_tune, y_val_tune),
             callbacks=[tuner_early_stopping],
             verbose=1)

# Get the optimal hyperparameters
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print(f"""
--- Hyperparameter Search Complete ---
Best Hyperparameters Found:
- LSTM Layer 1 Units: {best_hps.get('units_1')}
- Dropout 1 Rate: {best_hps.get('dropout_1'):.2f}
- LSTM Layer 2 Units: {best_hps.get('units_2')}
- Dropout 2 Rate: {best_hps.get('dropout_2'):.2f}
- Dense Layer Units: {best_hps.get('dense_units')}
- Learning Rate: {best_hps.get('learning_rate')}
- Batch Size: {best_hps.get('batch_size')}
""") 

Trial 10 Complete [00h 01m 10s]
val_loss: 0.00018567102233646438

Best val_loss So Far: 0.00016844100173329934
Total elapsed time: 00h 13m 25s

--- Hyperparameter Search Complete ---
Best Hyperparameters Found:
- LSTM Layer 1 Units: 32
- Dropout 1 Rate: 0.00
- LSTM Layer 2 Units: 128
- Dropout 2 Rate: 0.20
- Dense Layer Units: 32
- Learning Rate: 0.01
- Batch Size: 64



## 9. Train Final Initial Model with Best Hyperparameters

In [9]:
print("\n--- Training Final Initial LSTM Model ---")
start_time_initial_train = time.time()

# Prepare sequences using the *entire* initial train+validation data
scaled_train_val_data = scaled_data[:n_train_val]
x_train_val_final, y_train_val_final = create_sequences(scaled_train_val_data, look_back)
x_train_val_final = np.reshape(x_train_val_final, (x_train_val_final.shape[0], x_train_val_final.shape[1], 1))

# Build the final model with the best hyperparameters directly from tuner
final_lstm_model = tuner.hypermodel.build(best_hps)  

# Define early stopping for the final training phase
final_early_stopping = callbacks.EarlyStopping(
    monitor='loss',  
    patience=FINAL_TRAINING_PATIENCE,
    restore_best_weights=True  
)


print(f"Training final initial LSTM on {n_train_val} data points for up to {FINAL_TRAINING_EPOCHS} epochs...")
history_final = final_lstm_model.fit(
    x_train_val_final, y_train_val_final,
    epochs=FINAL_TRAINING_EPOCHS,
    batch_size=best_hps.get('batch_size') if 'batch_size' in best_hps else FINAL_TRAINING_BATCH_SIZE, 
    callbacks=[final_early_stopping],
    verbose=1 
)

end_time_initial_train = time.time()
print(f"Final Initial LSTM Training Complete in {end_time_initial_train - start_time_initial_train:.2f} seconds.")
final_lstm_model.summary()


--- Training Final Initial LSTM Model ---
Training final initial LSTM on 1461 data points for up to 100 epochs...
Epoch 1/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 4s 71ms/step - loss: 2.2237
Epoch 2/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 79ms/step - loss: 0.0088
Epoch 3/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 74ms/step - loss: 0.0033
Epoch 4/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 3s 124ms/step - loss: 0.0025
Epoch 5/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 83ms/step - loss: 0.0026
Epoch 6/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 88ms/step - loss: 0.0021
Epoch 7/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 79ms/step - loss: 0.0021
Epoch 8/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 3s 141ms/step - loss: 0.0018
Epoch 9/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 100ms/step - loss: 0.0020
Epoch 10/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 88ms/step - loss: 0.0018
Epoch 11/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 3s 119ms/step - loss: 0.0017
Epoch 12/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 69ms/step - loss: 0.0017
Epoch 13/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 80ms/step - loss: 0.0016
Epoch 

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 60, 32)         │         4,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 60, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 128)            │        82,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         4,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 272,837 (1.04 MB)

 Trainable params: 90,945 (355.25 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 181,892 (710.52 KB)

## 10. Walk-Forward Validation (Rolling Forecast) Loop

In [10]:
print(f"\n--- Starting LSTM Walk-Forward Validation for {n_test} steps ---")
start_time_walk_forward = time.time()

lstm_walk_forward_predictions = [] 
# Initialize history with all scaled data up to the start of the test set
history_scaled = scaled_data[:n_train_val].flatten().tolist()

for i in range(n_test):
    # 1. Define the exact index for the current step in the full dataset
    current_full_index = n_train_val + i

    # 2. Prepare the input sequence
    if len(history_scaled) < look_back:
         raise IndexError("History length too short for look_back.")
    input_sequence = np.array(history_scaled[-look_back:]).reshape((1, look_back, 1))

    # 3. Predict 1-step ahead (scaled) using the *final_lstm_model*
    pred_scaled = final_lstm_model.predict(input_sequence, verbose=0)[0, 0]

    # 4. Inverse transform the prediction
    pred_unscaled = scaler.inverse_transform([[pred_scaled]])[0, 0]
    lstm_walk_forward_predictions.append(pred_unscaled)

    # Update History with actual value ---
    # 5. Get the actual scaled value for the current time step `i`
    actual_scaled_value_i = scaled_data[current_full_index, 0]

    # 6. Append the *actual* scaled value to the history
    history_scaled.append(actual_scaled_value_i)

    # Periodic Retraining
    if RETRAIN_FREQUENCY > 0 and (i + 1) % RETRAIN_FREQUENCY == 0 and (i + 1) < n_test : 
        print(f"\n--- Retraining LSTM at step {i+1}/{n_test} ---")
        retrain_start_time = time.time()
        current_history_array = np.array(history_scaled).reshape(-1, 1)
        x_retrain, y_retrain = create_sequences(current_history_array, look_back)
        x_retrain = np.reshape(x_retrain, (x_retrain.shape[0], x_retrain.shape[1], 1))

        # Retrain the existing model for a few epochs
        print(f"Retraining on {len(x_retrain)} sequences...")
        final_lstm_model.fit(x_retrain, y_retrain,
                             epochs=RETRAIN_EPOCHS,
                             batch_size=best_hps.get('batch_size') if 'batch_size' in best_hps else FINAL_TRAINING_BATCH_SIZE,
                             verbose=0) 
        retrain_end_time = time.time()
        print(f"Retraining complete in {retrain_end_time - retrain_start_time:.2f} seconds.")
        # End Retraining

    # Log progress
    elif (i + 1) % 100 == 0:
        print(f"LSTM Walk-Forward Step {i+1}/{n_test} complete.")


end_time_walk_forward = time.time()
total_walk_forward_time = end_time_walk_forward - start_time_walk_forward
print(f"\nLSTM Walk-Forward finished in {total_walk_forward_time:.2f} seconds.")

# Ensure predictions list is numpy array
lstm_walk_forward_predictions = np.array(lstm_walk_forward_predictions)


--- Starting LSTM Walk-Forward Validation for 366 steps ---
LSTM Walk-Forward Step 100/366 complete.
LSTM Walk-Forward Step 200/366 complete.
LSTM Walk-Forward Step 300/366 complete.

LSTM Walk-Forward finished in 18.36 seconds.


## 11. Evaluate Walk-Forward Performance

In [11]:
# Define the evaluation metrics function 
def evaluate_forecast(y_true, y_pred, model_name):
    """Calculates and prints standard evaluation metrics."""
    y_true_flat = y_true.flatten(); y_pred_flat = y_pred.flatten()
    mae = mean_absolute_error(y_true_flat, y_pred_flat)
    mape = mean_absolute_percentage_error(y_true_flat, y_pred_flat)
    rmse = np.sqrt(mean_squared_error(y_true_flat, y_pred_flat))
    try: r2 = r2_score(y_true_flat, y_pred_flat)
    except ValueError: r2 = np.nan
    print(f"\n--- {model_name} Walk-Forward (t+1) Evaluation Results ---")
    print(f"RMSE: {rmse:.4f}, MAE: {mae:.4f}, MAPE: {mape:.4%}, R²: {r2:.4f}")
    return {'RMSE': rmse, 'MAE': mae, 'MAPE': mape, 'R2': r2}

# Evaluate against the actual unscaled test data
y_test_actual = test_data_df['Close'].values
if len(y_test_actual) != len(lstm_walk_forward_predictions):
    raise ValueError(f"Length mismatch: Actual test data ({len(y_test_actual)}) vs Predictions ({len(lstm_walk_forward_predictions)})")

lstm_wf_results = evaluate_forecast(y_test_actual, lstm_walk_forward_predictions, f"LSTM ({ticker})")


--- LSTM (BTC-USD) Walk-Forward (t+1) Evaluation Results ---
RMSE: 2516.2393, MAE: 1856.5922, MAPE: 2.6782%, R²: 0.9706


## 12. Visualize Walk-Forward Results

In [12]:
print("\n--- Plotting Walk-Forward Forecasts ---")
results_df_wf = pd.DataFrame({
    'Actual': y_test_actual.flatten(),
    f'LSTM (t+1)': lstm_walk_forward_predictions.flatten()
}, index=test_data_df.index)

fig = go.Figure()
fig.add_trace(go.Scatter(x=results_df_wf.index, y=results_df_wf['Actual'], mode='lines', name='Actual Price (Test)', line=dict(color='black')))
fig.add_trace(go.Scatter(x=results_df_wf.index, y=results_df_wf[f'LSTM (t+1)'], mode='lines', name='LSTM Walk-Forward (t+1)', line=dict(color='green', dash='dash')))
fig.update_layout(
    title=f'LSTM Walk-Forward (t+1) Forecast Comparison for {ticker} (Tuned)',
    xaxis_title="Date", yaxis_title="Price (USD)", legend_title="Data/Model", template="plotly_white"
)
fig.show()


--- Plotting Walk-Forward Forecasts ---


## 13. Walk-Forward Evaluation Period Summary

In [13]:
print(f"\n--- Walk-Forward Evaluation Summary ---")
print(f"Initial Train+Validation Data End Date: {train_val_data_df.index.max().strftime('%Y-%m-%d')}")
print(f"Walk-Forward Evaluation Period (Test Set): {test_data_df.index.min().strftime('%Y-%m-%d')} to {test_data_df.index.max().strftime('%Y-%m-%d')}")
print(f"Number of Walk-Forward Steps (Predictions): {n_test}")


--- Walk-Forward Evaluation Summary ---
Initial Train+Validation Data End Date: 2023-12-31
Walk-Forward Evaluation Period (Test Set): 2024-01-01 to 2024-12-31
Number of Walk-Forward Steps (Predictions): 366
